In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler

In [2]:
df = pd.read_excel("fraud_dataset.xlsx")

In [6]:
X = df.drop("target", axis=1).values
y=df["target"].values

scaler = StandardScaler()
X = scaler.fit_transform(X)

X = torch.tensor(X,dtype=torch.float32)
y = torch.tensor(y,dtype=torch.float32).view(-1,1)


class FraudNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(5, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.sigmoid(self.fc3(x))
        return x

model = FraudNet()

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 4000

for epoch in range(epochs):
    y_pred = model(X)
    loss = criterion(y_pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if(epoch+1)%400==0:
        print(f"Epoch {epoch+1} Loss {loss.item():.4f}")

with torch.no_grad():
    probs = model(X)
    preds = (probs>0.5).float()

    accuracy = (preds == y).float().mean() * 100
    print(f"\nTrain Accuracy = {accuracy:.2f}%")

new_transaction = torch.tensor(
    [[3200, 2, 18, 1, 2]],
    dtype=torch.float32
)

new_transaction = torch.tensor(
    scaler.transform(new_transaction.numpy()),
    dtype=torch.float32
)

with torch.no_grad():
    prob = model(new_transaction).item()

    result = "FRAUD" if prob > 0.5 else "NORMAL"

    print(f"Risk = {prob * 100:.2f}%")
    print(f"Prediction = {result}")




Epoch 400 Loss 0.0799
Epoch 800 Loss 0.0504
Epoch 1200 Loss 0.0354
Epoch 1600 Loss 0.0259
Epoch 2000 Loss 0.0168
Epoch 2400 Loss 0.0086
Epoch 2800 Loss 0.0043
Epoch 3200 Loss 0.0024
Epoch 3600 Loss 0.0014
Epoch 4000 Loss 0.0009

Train Accuracy = 100.00%
Risk = 100.00%
Prediction = FRAUD
